# Lab 27 — Optimize a Streaming Response System Using FastAPI SSE

**Module:** Performance Optimization · Day 15 · Session 02
**Duration:** ~45-55 minutes

### What you will do
Add a Server-Sent Events streaming endpoint to Day 14 Lab 24's FastAPI
server, and measure the real Time to First Token (TTFT) improvement
against the original non-streaming endpoint.

### Prerequisite
Lab 24's FastAPI + JWT auth code should be available to reload.

## Step 1 — Reload Lab 24's FastAPI app + JWT auth

In [ ]:
%pip install -q fastapi pyjwt uvicorn requests python-multipart openai

In [ ]:
import os, json, time, threading
from datetime import datetime, timedelta
from fastapi import FastAPI, Depends, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from pydantic import BaseModel
import jwt, uvicorn, requests
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before continuing"
client = OpenAI()

app = FastAPI(title="ClaimsIQ Streaming API")
SECRET_KEY = os.environ.get("CLAIMSIQ_JWT_SECRET", "dev-only-secret-change-in-production")
ALGORITHM = "HS256"
MOCK_USERS = {"claims_agent": "demo_password"}
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")

@app.post("/token")
def login(form_data: OAuth2PasswordRequestForm = Depends()):
    if MOCK_USERS.get(form_data.username) != form_data.password:
        raise HTTPException(status_code=401, detail="Incorrect username or password")
    expire = datetime.utcnow() + timedelta(minutes=30)
    token = jwt.encode({"sub": form_data.username, "exp": expire}, SECRET_KEY, algorithm=ALGORITHM)
    return {"access_token": token, "token_type": "bearer"}

def verify_token(token: str = Depends(oauth2_scheme)) -> dict:
    try:
        return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
    except jwt.PyJWTError:
        raise HTTPException(status_code=401, detail="Invalid or expired token")

print("Base app + auth reloaded from Lab 24.")

## Step 2 — Add the ORIGINAL non-streaming endpoint (for comparison)

In [ ]:
class ClaimRequest(BaseModel):
    customer_id: str
    claim_id: str

@app.post("/process_claim")
def process_claim(req: ClaimRequest, user: dict = Depends(verify_token)):
    prompt = f"Explain in 3-4 sentences why a warranty claim for order {req.claim_id} might be approved or denied. Be specific and detailed."
    response = client.chat.completions.create(model="gpt-4o-mini", messages=[{"role": "user", "content": prompt}], temperature=0)
    return {"decision_reasoning": response.choices[0].message.content}

print("/process_claim (non-streaming) defined.")

## Step 3 — Write `event_generator()` with `stream=True`

In [ ]:
def event_generator(customer_id: str, claim_id: str):
    prompt = f"Explain in 3-4 sentences why a warranty claim for order {claim_id} might be approved or denied. Be specific and detailed."
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        stream=True,
    )
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            yield f"data: {delta}\n\n"
    yield "data: [DONE]\n\n"

print("event_generator() ready.")

## Step 4 — Add `/process_claim_stream` with `StreamingResponse`

In [ ]:
@app.get("/process_claim_stream")
def process_claim_stream(customer_id: str, claim_id: str, user: dict = Depends(verify_token)):
    return StreamingResponse(event_generator(customer_id, claim_id), media_type="text/event-stream")

print("/process_claim_stream defined.")

## Step 5 — Start the server, get a JWT token

In [ ]:
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8900, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

token_resp = requests.post("http://127.0.0.1:8900/token", data={"username": "claims_agent", "password": "demo_password"})
token = token_resp.json()["access_token"]
headers = {"Authorization": f"Bearer {token}"}
print("Server running, token obtained.")

## Step 6 — Call the NON-streaming endpoint, time TTFT

For a non-streaming endpoint, "TTFT" is really just total latency,
since nothing is visible until the entire response arrives.

In [ ]:
start = time.time()
resp = requests.post(
    "http://127.0.0.1:8900/process_claim",
    json={"customer_id": "CUST99001", "claim_id": "CLM99001"},
    headers=headers,
)
non_streaming_ttft = time.time() - start
print(f"Non-streaming TTFT (= total latency): {non_streaming_ttft:.2f}s")
print("Response:", resp.json()["decision_reasoning"][:100], "...")

## Step 7 — Call the streaming endpoint, time REAL TTFT

This time, TTFT is the moment the FIRST chunk of text actually arrives
— not when the whole thing finishes.

In [ ]:
start = time.time()
first_chunk_time = None
full_text = ""

with requests.get(
    "http://127.0.0.1:8900/process_claim_stream",
    params={"customer_id": "CUST99001", "claim_id": "CLM99001"},
    headers=headers,
    stream=True,
) as r:
    for line in r.iter_lines():
        if line:
            decoded = line.decode("utf-8")
            if decoded.startswith("data: ") and "[DONE]" not in decoded:
                if first_chunk_time is None:
                    first_chunk_time = time.time()
                full_text += decoded.replace("data: ", "")

streaming_ttft = first_chunk_time - start
total_streaming_time = time.time() - start
print(f"Streaming TTFT (first chunk):  {streaming_ttft:.2f}s")
print(f"Streaming total time:          {total_streaming_time:.2f}s")

## Step 8 — Compare the two TTFT measurements

In [ ]:
print("=" * 50)
print(f"Non-streaming TTFT: {non_streaming_ttft:.2f}s  (= total time)")
print(f"Streaming TTFT:      {streaming_ttft:.2f}s  (first visible content)")
print(f"Streaming total:     {total_streaming_time:.2f}s  (roughly the same total time as non-streaming)")
print("=" * 50)
print(f"\nPerceived latency improvement: {non_streaming_ttft - streaming_ttft:.2f}s faster to first visible output")

## Step 9 — Confirm an invalid token still gets rejected on the streaming endpoint

In [ ]:
bad_resp = requests.get(
    "http://127.0.0.1:8900/process_claim_stream",
    params={"customer_id": "CUST99001", "claim_id": "CLM99001"},
    headers={"Authorization": "Bearer not-a-real-token"},
)
print("Status without a valid token:", bad_resp.status_code)
assert bad_resp.status_code == 401, "Streaming endpoint should still require valid auth!"
print("PASS — Day 14's JWT auth applies identically to the streaming endpoint.")

## Deliverable

1. The TTFT comparison from Step 8.
2. Confirmation from Step 9 that auth still applies to the streaming
   route.
3. One paragraph: notice `total_streaming_time` in Step 7 is roughly
   the SAME as `non_streaming_ttft` from Step 6 — streaming didn't make
   the underlying work any faster. Explain, in your own words, exactly
   what streaming DID improve, and what it explicitly did NOT improve.